## Funciones de libreria REmatch

Ya contamos con todos los operadores más importantes de expresiones regulares que están incluidos en REQL (de hecho, todos los ejemplos anteriores son patrones/consultas en el lenguaje REQL). Existen otros operadores o extensiones que nos pueden permitir simplificar nuestros patrones, pero las vistas anteriormente son las principales, y con ellas podemos hacer la mayoría de las tareas.

Ahora veremos como utilizar REQL para, no solo verificar si se cumple un patrón o no, si no también para extraer las partes del documento que cumplen con este patrón. Para esto, usaremos consultas REQL y las extenderemos con variables que almacenarán las partes del string o documento que cumplen con el patrón.

### Variables y captura

Para entender como usar REQL y extraer partes de nuestro strings, veamos un ejemplo. Considere nuestro patrón sencillo para verificar un correo, solo que ahora le agregaremos variables para especificar las partes que queremos extraer:

`^!x{(\w+\.)?\w+}@!y{(\w+\.)?\w+\.\w{2,3}}$`

Nuestro nuevo patrón cuenta con dos variables. La primera variable, escrita como `!x{...}` nos permitirá obtener el identificador del correo, mientras que la segunda variable, escrita como `!y{...}` nos permitirá obtener el dominio del correo. De hecho, el identificador `x` e `y` son los nombres de estas variables, y nos permitirá referirnos por un nombre a cada captura. Estos nombres son arbitrarios, y podríamos haber ocupado identificadores más sugerentes como a continuación:

`^!id{(\w+\.)?\w+}@!domain{(\w+\.)?\w+\.\w{2,3}}$`

donde `id` y `domain` son los nuevos identificadores para `x` e `y`, respectivamente.

Ahora que tenemos nuestras variables para capturar contenido, necesitaremos ver como utilizar la librería REmatch para extraer el contenido. Para esto, veamos un ejemplo en ejecución.

In [2]:
import pyrematch as REmatch

In [5]:
seq = ["cperez@gmail.com", "soto@uc.cl", "sdelcampo@gmail.com",
       "lpalacios@gmeil.com", "rramirez@gmsil.com", "pvergara@ing.uc.cl",
       "ndelafuente@ing.puc.cl", "tnovoa@mail.uc.cl", "nnarea@myucmail.uc.cl",
       "nomail@gmail.com", "juan.soto@uc.cl"]
pattern = r"^!id{(\w+\.)?\w+}@!domain{(\w+\.)?\w+\.\w{2,3}}$"
query = REmatch.reql(pattern)

for s in seq:
    match = query.findone(s)
    if match:
        print("El correo " + s +
              " tiene id " + match.group('id') +
              " y dominio " + match.group('domain'))

El correo cperez@gmail.com tiene id cperez y dominio gmail.com
El correo soto@uc.cl tiene id soto y dominio uc.cl
El correo sdelcampo@gmail.com tiene id sdelcampo y dominio gmail.com
El correo lpalacios@gmeil.com tiene id lpalacios y dominio gmeil.com
El correo rramirez@gmsil.com tiene id rramirez y dominio gmsil.com
El correo pvergara@ing.uc.cl tiene id pvergara y dominio ing.uc.cl
El correo ndelafuente@ing.puc.cl tiene id ndelafuente y dominio ing.puc.cl
El correo tnovoa@mail.uc.cl tiene id tnovoa y dominio mail.uc.cl
El correo nnarea@myucmail.uc.cl tiene id nnarea y dominio myucmail.uc.cl
El correo nomail@gmail.com tiene id nomail y dominio gmail.com
El correo juan.soto@uc.cl tiene id juan.soto y dominio uc.cl


Como se puede observar de este ejemplo, la función `findone` del objeto query nos devuelve un objeto `match` cuando obtiene un resultado. Este objeto `match` tiene la captura con el contenido de `id` y de `domain`. Para acceder el string capturado por estas variables, usamos la función `group` con el nombre de la variable.

### Extracción de información en documentos

Suponga ahora que en vez de un solo string, usted cuenta con un documento que es una lista de string separados por comas, donde algunos de estos strings pueden ser correos y otros no. Por ejemplo, un posible documento con estas caracteristicas sería el siguiente:

`Carlos Perez,cperez@gmail.com,Juan Soto,soto@uc.cl,Sebastian del Campo,sdelcampo@gmail.com`

Usted desea extraer los strings que son correos, junto con su identificador y dominio. Si extendemos ligeramente nuestro patrón REQL podremos identificar si un campo en los datos (esto es, un string separado por comas) es un correo y, utilizando las variables, podremos obtener el correo, el identificador y el dominio. En otras palabras, considere la siguiente consulta REQL:

`(^|,)!email{!id{(\w+\.)?\w+}@!domain{(\w+\.)?\w+\.\w{2,3}}}($|,)`

Lo que hicimos fue modificar el comienzo con `(^|,)` que nos permite partir desde el comienzo del string o desde una coma, y también modificar el fin con `($|,)` que nos permite terminar desde el fin del string o desde una coma.
Esta consulta REQL busca un correo en el documento y, de encontrarlo, obtiene el correo completo, el identificador y dominio del correo.

Volviendo a nuestro objetivo, nosotros queremos obtener todos los correos en esta lista y no solo saber si hay un correo o no. Para esto, al evaluar nuestra nueva consulta la función `finditer` de REmatch buscará todos los posibles **matches** de nuestro patrón en el documento, permitiendo recorrerlos uno a uno. Veamos un ejemplo de como funciona `finditer`.

In [7]:
document = 'Carlos Perez,cperez@gmail.com,Juan Soto,soto@uc.cl,Sebastian del Campo,sdelcampo@gmail.com'
pattern = r"(^|,)!email{!id{(\w+\.)?\w+}@!domain{(\w+\.)?\w+\.\w{2,3}}}($|,)"
query = REmatch.reql(pattern)

for match in query.finditer(document):
    print("El correo " + match.group('email') +
          " tiene id " + match.group('id') +
          " y dominio " + match.group('domain'))

El correo cperez@gmail.com tiene id cperez y dominio gmail.com
El correo soto@uc.cl tiene id soto y dominio uc.cl
El correo sdelcampo@gmail.com tiene id sdelcampo y dominio gmail.com


### Superposición de resultados

Es importante notar que REmatch buscará todas las ocurrencias de nuestro patrón en el documento, posiblemente con contenidos repetidos en el documento. Para entender esto, tratemos ahora de buscar todos los campos que estan entre comas con la consulta REQL:

`(^|,)!text{.+}($|,)`

Este patrón buscara el comienzo del documento o una coma, seguido de uno o más caracteres, y terminado en el fin del documento o una coma. Por último la secuencia de uno o más caracteres será almacenado en la variable `text`. Comprobemos ahora como funciona este patrón REQL en REmatch y que será lo que entregará la función `finditer`.

In [8]:
document = 'Carlos Perez,cperez@gmail.com,Juan Soto,soto@uc.cl,Sebastian del Campo,sdelcampo@gmail.com'
pattern = "(^|,)!text{.+}($|,)"
query = REmatch.reql(pattern)

for match in query.finditer(document):
    print("El texto encontrado fue " + match.group('text'))

El texto encontrado fue Carlos Perez
El texto encontrado fue cperez@gmail.com
El texto encontrado fue Carlos Perez,cperez@gmail.com
El texto encontrado fue Juan Soto
El texto encontrado fue cperez@gmail.com,Juan Soto
El texto encontrado fue Carlos Perez,cperez@gmail.com,Juan Soto
El texto encontrado fue soto@uc.cl
El texto encontrado fue Juan Soto,soto@uc.cl
El texto encontrado fue cperez@gmail.com,Juan Soto,soto@uc.cl
El texto encontrado fue Carlos Perez,cperez@gmail.com,Juan Soto,soto@uc.cl
El texto encontrado fue Sebastian del Campo
El texto encontrado fue soto@uc.cl,Sebastian del Campo
El texto encontrado fue Juan Soto,soto@uc.cl,Sebastian del Campo
El texto encontrado fue cperez@gmail.com,Juan Soto,soto@uc.cl,Sebastian del Campo
El texto encontrado fue Carlos Perez,cperez@gmail.com,Juan Soto,soto@uc.cl,Sebastian del Campo
El texto encontrado fue sdelcampo@gmail.com
El texto encontrado fue Sebastian del Campo,sdelcampo@gmail.com
El texto encontrado fue soto@uc.cl,Sebastian del Camp

Puede ver que REmatch encontro todos los campos entre comas, pero tambien campos que cruzan entre una o más comas. Recuerde que `.+` buscará calzar con "uno o más" caracteres y como la coma es un carácter, también cumple con lo especificado. Si queremos encontrar todos los campos, entonces tenemos que decirle a REmatch que encuentre uno o más caracteres que **no** sea una coma. Esto ya lo sabemos hacer con la clase `[^,]`, por lo cual, el patrón que necesitamos será el siguiente:

`(^|,)!text{[^,]+}($|,)`

Si probamos este patrón en el documento anterior, podremos ver que ahora si encontraremos solo los campos entre dos comas consecutivas.

In [10]:
document = 'Carlos Perez,cperez@gmail.com,Juan Soto,soto@uc.cl,Sebastian del Campo,sdelcampo@gmail.com'
pattern = "(^|,)!text{[^,]+}($|,)"
query = REmatch.reql(pattern)

for match in query.finditer(document):
    print('El campo encontrado fue "{0}"'.format(match.group('text')))

El campo encontrado fue "Carlos Perez"
El campo encontrado fue "cperez@gmail.com"
El campo encontrado fue "Juan Soto"
El campo encontrado fue "soto@uc.cl"
El campo encontrado fue "Sebastian del Campo"
El campo encontrado fue "sdelcampo@gmail.com"


### Groups versus spans

Hasta ahora hemos extraido el contenido de cada captura. Muchas veces también nos interesa saber la posición donde aparece ese contenido, más que el contenido mismo. La posición del contenido en el documento viene dado por lo que se conoce como un *span*, que es un par `(i, j)` con `i` menor que `j`. Este span `(i,j)` define un intervalo del string, o sea, define el contenido desde el carácter `i` hasta el carácter `j`. Si volvemos a nuestro ejemplo anterior de campos de un documento, podemos extraer las posiciones de donde aparce cada campo `text`, utilizando el método `span` de un match.

In [11]:
document = 'Carlos Perez,cperez@gmail.com,Juan Soto,soto@uc.cl,Sebastian del Campo,sdelcampo@gmail.com'
pattern = "(^|,)!text{[^,]+}($|,)"
query = REmatch.reql(pattern)
for match in query.finditer(document):
    print('El campo encontrado fue "{0}" en la posición {1}'.format(match.group('text'), match.span('text')))

El campo encontrado fue "Carlos Perez" en la posición (0, 12)
El campo encontrado fue "cperez@gmail.com" en la posición (13, 29)
El campo encontrado fue "Juan Soto" en la posición (30, 39)
El campo encontrado fue "soto@uc.cl" en la posición (40, 50)
El campo encontrado fue "Sebastian del Campo" en la posición (51, 70)
El campo encontrado fue "sdelcampo@gmail.com" en la posición (71, 90)


Es importante notar que el span de un match nos dice el contexto donde aparece el contenido, y esto nos puede ayudar para saber si este aparece al comienzo del documento, al final, etc. Para manejo de spans y groups, el objeto `match` cuenta con las siguientes funcionalidades:

- `start(id-variable)`: entrega la posición inicial del span de la variable con nombre `id-variable`.
- `end(id-variable)`: entrega la posición final del span de la variable con nombre `id-variable`.

# ¿Por qué utilizamos la librería REmatch?

REmatch es una librería para extracción de información desarrollada por investigadores y alumnos del Departamento de Ciencia de la Computación de la Escuela de Ingeniería. Esta librería cuenta con varias ventajas con respecto a otras librerías.

REmatch es una librería orientada a extraer información desde archivos de texto grandes, y no solo a la búsqueda de un patron adentro del texto. En este contexto, REmatch permite extraer *todos* los matches, y no solo encontrar el primer match. Con todos los matches nos referimos que REmatch también puede detectar matches solapados, que es algo fuera del alcance de otras librerías, y muy útil en varios contextos, cómo, por ejemplo, en el análisis de secuencias de ADN, procesamiento de logs, texto natural, etc.

Adicionalmente, REmatch es orientado a procesar documentos grandes de manera muy rápida, y extraer información de manera muy eficiente. Por lo tanto, REmatch evita las debilidades de varias otras librerías de expresiones regulares, y no permite ataques de estilo ReDos (ver https://en.wikipedia.org/wiki/ReDoS), a los cuales padecen muchas librerías clasicas de expresiones regulares.

Para aprender más sobre la librería REmatch, te invitamos visitar nuestra página web https://rematch.cl donde podrás encontrar tutoriales y ejemplos de como escribir patrones en REQL y probarlos directamente. También te invitamos a visitar el github https://github.com/REmatchChile/REmatch/ o su wiki https://github.com/REmatchChile/REmatch/wiki.